In [ ]:
# Split all prediction results by basin, export one separate csv file for each basin
import pandas as pd
import os

# Input and output paths
input_file = r'D:\DESKTOP\desk\bianliang\\esm-ssp585\\p12\\p12ALL.csv'
output_folder = r'D:\DESKTOP\desk\bianliang\\esm-ssp585\\attribution\\esm-ssp585-ALL-basins'

os.makedirs(output_folder, exist_ok=True)

df = pd.read_csv(input_file)

columns_to_save = ['year', 'PPT', 'PET', 'R', 'n', 'AET', 'LAI', 'treeFrac']

# Group by the first column (basin identifier)
for no_value, group in df.groupby(df.columns[0]):
    group_selected = group[columns_to_save]
    output_path = os.path.join(output_folder, f"{no_value}.csv")
    group_selected.to_csv(output_path, index=False)

print("All basin‑level files have been saved to:", output_folder)

In [ ]:
# For spatial mapping
# Attribute the last window (change period 2071--2100) for all basins, and calculate decadal variation and change rate (2031--2040, 2061--2070, 2091--2100)
import pandas as pd
import numpy as np
import math
import os

# Budyko model function
def budyko(PPT, PET, n):
    phi = PET / PPT
    ET = PPT * (1 + phi - (1 + phi**n)**(1/n))
    R = PPT - ET
    return R

# Elasticity coefficient calculation
def compute_elasticities(PPT, PET, n):
    phi = PET / PPT
    eps_ppt = ((1 + phi**n)**(1/n+1) - phi**(n+1)) / ((1 + phi**n) * ((1 + phi**n)**(1/n) - phi))
    eps_pet = 1 / ((1 + phi**n) * (1 - (1 + phi**-n)**(1/n)))
    eps_n = (math.log(1 + phi**n) + phi**n * math.log(1 + phi**-n)) / (n * (1 + phi**n) * (1 - (1 + phi**-n)**(1/n)))
    return eps_ppt, eps_pet, eps_n

# Path configuration
input_folder = r'D:\DESKTOP\desk\bianliang\\esm-ssp585\\attribution\\esm-ssp585-ALL-basins'
output_file = r'D:\DESKTOP\desk\bianliang\\esm-ssp585\\attribution\\guiyin\\esm-ssp585-ALL.csv'

all_results = []

for filename in os.listdir(input_folder):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_folder, filename)
        df = pd.read_csv(file_path)

        # Full time period average
        all_period = df[(df['year'] >= 1985) & (df['year'] <= 2100)]
        PPT00 = all_period['PPT'].mean()
        PET00 = all_period['PET'].mean()
        n00 = all_period['n'].mean()
        R00 = all_period['R'].mean()
        eps_ppt00, eps_pet00, eps_n00 = compute_elasticities(PPT00, PET00, n00)

        # Baseline period
        base_period = df[(df['year'] >= 1985) & (df['year'] <= 2014)]
        PPT0 = base_period['PPT'].mean()
        PET0 = base_period['PET'].mean()
        n0 = base_period['n'].mean()
        R0 = base_period['R'].mean()
        tree_base = base_period['treeFrac'].mean()
        hisR = R0

        # Change period
        change_period = df[(df['year'] >= 2071) & (df['year'] <= 2100)]
        PPT1 = change_period['PPT'].mean()
        PET1 = change_period['PET'].mean()
        n1 = change_period['n'].mean()
        R1 = change_period['R'].mean()
        dis_n = n1 - n0
        eps_ppt1, eps_pet1, eps_n1 = compute_elasticities(PPT1, PET1, n1)

        # ΔR decomposition
        delta_PPT = PPT1 - PPT0
        delta_PET = PET1 - PET0
        delta_n = n1 - n0
        delta_RPPT = delta_PPT / PPT00 * eps_ppt00 * R00
        delta_RPET = delta_PET / PET00 * eps_pet00 * R00
        delta_Rn = delta_n / n00 * eps_n00 * R00
        delta_RCC = delta_RPPT + delta_RPET
        delta_R_total = delta_RPPT + delta_RPET + delta_Rn

        if delta_R_total != 0:
            delta_PPT_percent = delta_RPPT / delta_R_total * 100
            delta_PET_percent = delta_RPET / delta_R_total * 100
            delta_CC_percent = delta_PPT_percent + delta_PET_percent
            delta_n_percent = delta_Rn / delta_R_total * 100
        else:
            delta_PPT_percent = delta_PET_percent = delta_n_percent = np.nan
            delta_CC_percent = np.nan

        # Time windows
        windows = {
            '4': (2031, 2040),
            '7': (2061, 2070),
            '10': (2091, 2100),
        }

        base_vars = base_period[['LAI', 'PPT', 'AET', 'R']].mean()
        variation_values = []
        rate_values = []
        tree_values = []
        tree_deltas = []

        for label, (start, end) in windows.items():
            future_period = df[(df['year'] >= start) & (df['year'] <= end)]
            if future_period.empty:
                variation_values.extend([np.nan] * 4)
                rate_values.extend([np.nan] * 4)
                tree_values.append(np.nan)
                tree_deltas.append(np.nan)
            else:
                future_mean = future_period[['LAI', 'PPT', 'AET', 'R']].mean()
                delta = future_mean - base_vars
                rate = delta / base_vars * 100
                variation_values.extend(delta.values.tolist())
                rate_values.extend(rate.values.tolist())

                tree = future_period['treeFrac'].mean()
                tree_values.append(tree)
                tree_deltas.append(tree - tree_base)

        # Summary
        all_results.append([
            os.path.splitext(filename)[0],
            eps_ppt1, eps_pet1, eps_n1,
            delta_RPPT, delta_RPET, delta_RCC, delta_Rn, delta_R_total,
            delta_PPT_percent, delta_PET_percent, delta_CC_percent, delta_n_percent,
            hisR, dis_n
        ] + variation_values + rate_values + tree_values + tree_deltas)

# Construct column names
# eps_xx: elasticity; dRxx: runoff change induced by each factor (ΔRxx); pxx: contribution of each factor
# hisR: mean runoff in the historical period (1985--2014); dis_n: difference in n between the change period (2071--2100) and the baseline period (1985--2014)
columns = [
    'Filename',
    'eps_PPT', 'eps_PET', 'eps_n',
    'dRPPT', 'dRPET', 'dRCC', 'dRn', 'dR',
    'pPPT', 'pPET', 'pCC', 'pn',
    'hisR', 'dis_n'
]

# v_xx represents variation; r_xx represents change rate;
# tree_ represents forest cover; v_tree_xx represents forest cover variation;
# '4' stands for (2031, 2040); '7' stands for (2061, 2070); '10' stands for (2091, 2100)
vars_ = ['LAI', 'PPT', 'AET', 'R']
for label in ['4', '7', '10']:
    columns += [f'v_{label}_{v}' for v in vars_]
for label in ['4', '7', '10']:
    columns += [f'r_{label}_{v}' for v in vars_]
columns += [f'tree_{label}' for label in ['4', '7', '10']]
columns += [f'v_tree_{label}' for label in ['4', '7', '10']]

# Save results
output_df = pd.DataFrame(all_results, columns=columns)
output_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"All calculations completed! Results saved to {output_file}")

In [ ]:
# Write attributes into shapefile
import geopandas as gpd
import pandas as pd
import os

# Input paths
csv_file = r'D:\DESKTOP\desk\bianliang\\esm-ssp585\\attribution\\guiyin\\esm-ssp585-ALL.csv'
shp_folder = r'D:\DESKTOP\desk\\private\\map\BasinATLAS_Data_v10.gdb&shp\BasinATLAS5'
output_shp = r'D:\DESKTOP\desk\\private\\map\BasinATLAS_Data_v10.gdb&shp\\attribution\\esm-ssp585'

os.makedirs(output_shp, exist_ok=True)

df_csv = pd.read_csv(csv_file)

gdf_list = []

for idx, row in df_csv.iterrows():
    filename = row['Filename']
    shp_path = os.path.join(shp_folder, f"{filename}.shp")
    
    if not os.path.exists(shp_path):
        print(f"Warning: {shp_path} does not exist, skip this record.")
        continue
    
    gdf = gpd.read_file(shp_path)
    
    # Append all calculated attributes from csv to geodataframe
    for col in df_csv.columns[1:]: 
        gdf[col] = row[col]
    
    gdf_list.append(gdf)

if gdf_list:
    merged_gdf = gpd.GeoDataFrame(pd.concat(gdf_list, ignore_index=True), crs=gdf_list[0].crs)
    merged_gdf.to_file(output_shp, encoding='utf‑8‑sig')
    print(f"Task finished. Merged shapefile saved to: {output_shp}")
else:
    print("No valid shapefile was processed.")